# AI Phone Detection Model for Proctored Exam Engine
## YOLO11n – V2 (Partial Phone + Negative Images)

**Scope:** Phone detection model development and testing only.  
**Model:** YOLO11n | **Framework:** Ultralytics | **Environment:** Python, VS Code/Jupyter, Windows CPU

This notebook documents the V2 training configuration, evaluation metrics, and webcam inference. The model is already trained; **do not rerun the training cell unless intentionally retraining**.

## 1. Project Overview

The objective is to detect mobile phones visible in webcam frames during a proctored examination. V2 includes annotated partial/half-phone examples and phone-free negative images to improve robustness.

### Dataset summary
- Original train split: 529 images, 589 boxes
- Original validation split: 51 images, 69 boxes
- Original test split: 25 images, 27 boxes
- Added partial-phone images: 57
- Added negative images: 66
- V2 train dataset verification: 648 images and 648 label files

> Negative images have empty YOLO label files. Ensure an image is genuinely phone-free before treating it as a negative.

## 2. Dataset Configuration

Expected dataset layout:

```text
dataset/
└── yolo_phone/
    ├── train/
    │   ├── images/
    │   └── labels/
    ├── valid/
    │   ├── images/
    │   └── labels/
    ├── test/
    │   ├── images/
    │   └── labels/
    └── data.yaml
```

Example `data.yaml`:

```yaml
path: dataset/yolo_phone
train: train/images
val: valid/images
test: test/images
names:
  0: phone
```

Check the actual paths in your `data.yaml` before training.

In [7]:
import os
import cv2
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.15.0+cpu
CUDA available: False


In [8]:
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="harshdadiya-wappnet/phone_detection",
    repo_type="dataset",
    local_dir="dataset/hf_phone"
)

print("Dataset downloaded at:", dataset_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 610 files:   0%|          | 0/610 [00:00<?, ?it/s]

Dataset downloaded at: C:\Users\dell\Downloads\mobile_detection\dataset\hf_phone


In [9]:
from pathlib import Path

DATASET_PATH = Path("dataset/hf_phone")

for split in ["train", "valid", "test"]:
    folder = DATASET_PATH / split

    print("\n", split.upper())
    print("Folder exists:", folder.exists())

    if folder.exists():
        files = list(folder.iterdir())
        print("Total files:", len(files))
        print("Sample files:", [f.name for f in files[:5]])


 TRAIN
Folder exists: True
Total files: 530
Sample files: ['roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000031_jpg.rf.8301346c880a3546710c3d2d96272857.jpg', 'roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000031_jpg.rf.af209e900d50a6e98526d57eb997d549.jpg', 'roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000031_jpg.rf.b5ad1ede62e495edafadd275b8579a55.jpg', 'roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000035_jpg.rf.0b460f6dae8cd056b3a1d6f788e543de.jpg', 'roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000035_jpg.rf.931826d308be317d67db8eccee853a47.jpg']

 VALID
Folder exists: True
Total files: 52
Sample files: ['roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000021_jpg.rf.58ad653b3ecb06125cf393f5a85f669c.jpg', 'roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000124_jpg.rf.8eb4b61848d517f041089a2824fa1c73.jpg', 'roboflow_damaged_Refined-mobile-tablet-v1i-yolov11_000282_jpg.rf.1da1a8b8ae367684865643dea6504370.jpg', 'roboflow_damaged_Refined-mobile-tablet-v

In [10]:
import json

annotation_file = DATASET_PATH / "train" / "_annotations.coco.json"

with open(annotation_file, "r") as f:
    coco_data = json.load(f)

print("Images:", len(coco_data["images"]))
print("Annotations:", len(coco_data["annotations"]))
print("Categories:", coco_data["categories"])

print("\nSample annotation:")
print(coco_data["annotations"][0])

Images: 529
Annotations: 589
Categories: [{'id': 0, 'name': 'mobuile_phonw', 'supercategory': 'none'}, {'id': 1, 'name': 'mobile_phone', 'supercategory': 'mobuile_phonw'}]

Sample annotation:
{'id': 1, 'image_id': 0, 'category_id': 1, 'bbox': [64, 139, 384, 312], 'area': 119808, 'segmentation': [[448, 405.334, 384, 142.666, 98.666, 138.666, 64, 421.334, 81.334, 450.666, 448, 405.334]], 'iscrowd': 0}


In [11]:
from pathlib import Path

YOLO_DATA = Path("dataset/yolo_phone")

for split in ["train", "valid", "test"]:
    (YOLO_DATA / split / "images").mkdir(
        parents=True, exist_ok=True
    )
    (YOLO_DATA / split / "labels").mkdir(
        parents=True, exist_ok=True
    )

print("YOLO folders created!")

YOLO folders created!


In [12]:
import json
import shutil
from pathlib import Path

SOURCE_PATH = Path("dataset/hf_phone")
DEST_PATH = Path("dataset/yolo_phone")

def convert_coco_to_yolo(split):

    source_folder = SOURCE_PATH / split
    destination_folder = DEST_PATH / split

    annotation_path = source_folder / "_annotations.coco.json"

    with open(annotation_path, "r") as f:
        coco = json.load(f)

    # Image ID to filename mapping
    image_map = {
        image["id"]: image
        for image in coco["images"]
    }

    # Group annotations by image
    annotations_map = {}

    for ann in coco["annotations"]:
        image_id = ann["image_id"]

        annotations_map.setdefault(
            image_id, []
        ).append(ann)

    converted = 0

    for image_id, image_info in image_map.items():

        filename = image_info["file_name"]

        source_image = source_folder / filename

        if not source_image.exists():
            continue

        # Copy image
        shutil.copy2(
            source_image,
            destination_folder / "images" / filename
        )

        width = image_info["width"]
        height = image_info["height"]

        label_path = (
            destination_folder / "labels" /
            (Path(filename).stem + ".txt")
        )

        yolo_lines = []

        for ann in annotations_map.get(image_id, []):

            x, y, box_w, box_h = ann["bbox"]

            x_center = (x + box_w / 2) / width
            y_center = (y + box_h / 2) / height

            normalized_w = box_w / width
            normalized_h = box_h / height

            # Single class: mobile phone
            class_id = 0

            line = (
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{normalized_w:.6f} "
                f"{normalized_h:.6f}"
            )

            yolo_lines.append(line)

        # Save annotation file
        label_path.write_text(
            "\n".join(yolo_lines),
            encoding="utf-8"
        )

        converted += 1

    print(split, "images converted:", converted)


for split in ["train", "valid", "test"]:
    convert_coco_to_yolo(split)

train images converted: 529
valid images converted: 51
test images converted: 25


In [13]:
yaml_content = """
path: dataset/yolo_phone

train: train/images
val: valid/images
test: test/images

names:
  0: phone
"""

with open("dataset/yolo_phone/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created!")

data.yaml created!


In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11 nano  model (training v1)
model = YOLO("yolo11n.pt")

# Train on your custom phone dataset
results = model.train(
    data="dataset/yolo_phone/data.yaml",
    epochs=30,
    imgsz=416,
    batch=4,
    device="cpu",
    workers=0,
    project="runs/phone_detection",
    name="yolo11n_phone",
    patience=10,
    plots=True
)

In [14]:
from pathlib import Path

DATA_YAML = Path("dataset/yolo_phone/data.yaml")
print("data.yaml exists:", DATA_YAML.exists())
print("Current working directory:", Path.cwd())

data.yaml exists: True
Current working directory: c:\Users\dell\Downloads\mobile_detection


## 3. Training Configuration (V2)

The following is the original V2 training configuration. Training on CPU took approximately **346 minutes 57 seconds** in the recorded run.

**Warning:** Running this cell starts a new training run and can take several hours. It is included for reproducibility; skip it when using the existing `best.pt`.

In [ ]:
# OPTIONAL: Retraining cell — do not run unless you intentionally want to retrain.(training v2)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="dataset/yolo_phone/data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    device="cpu",
    workers=0,
    project="runs/phone_detection",
    name="phone_v2",
    patience=15,
    plots=True,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=0.5,
    close_mosaic=10
)

## 4. V1 vs V2 Evaluation Metrics

| Metric | V1 (normal phone) | V2 (partial phone + negatives) |
|---|---:|---:|
| Precision | 0.924 | 0.865 |
| Recall | 0.754 | 0.754 |
| mAP50 | 0.782 | 0.785 |
| mAP50–95 | 0.663 | 0.562 |

**Interpretation:** V2 mAP50 is marginally higher, while precision and mAP50–95 are lower. Recall is unchanged. Therefore, V2 is a candidate for partial-phone scenarios, not a model that improved on every metric. Validation was small (51 images / 69 instances), so these results are preliminary.

## 5. Load the Existing V2 Model

This cell loads the trained model; it does **not** train it.

If the path does not exist, locate the actual `best.pt` in your training run folder and update `MODEL_PATH`.

In [2]:
from pathlib import Path
from ultralytics import YOLO

MODEL_PATH = Path("runs/detect/runs/phone_detection/phone_v2-2/weights/best.pt")
print("Model file exists:", MODEL_PATH.exists())

model_v2 = YOLO(str(MODEL_PATH))

Model file exists: True


## 6. Single Image Inference (Optional)

Set `IMAGE_PATH` to an image you want to test. The model filters detections below the selected confidence threshold.

In [3]:
IMAGE_PATH = "webcam_test.jpg"
CONFIDENCE = 0.50

image_results = model_v2.predict(
    source=IMAGE_PATH,
    conf=CONFIDENCE,
    imgsz=416,
    verbose=False
)

image_results[0].show()
print("Detected phones:", len(image_results[0].boxes))

Detected phones: 0


## 7. Live Webcam Testing + Confidence Smoothing

The displayed confidence is a moving average of the highest phone confidence from up to five consecutive frames. This smooths the displayed score only; it does not improve model accuracy or change the underlying detection boxes.

Press **Q** to close the webcam window.

In [4]:
import cv2
from ultralytics import YOLO
from collections import deque

model_v2 = YOLO(
    "runs/detect/runs/phone_detection/phone_v2-2/weights/best.pt"
)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
CONFIDENCE = 0.50
confidence_history = deque(maxlen=5)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Webcam frame not received")
        break

    results = model_v2.predict(
        source=frame,
        conf=CONFIDENCE,
        imgsz=416,
        verbose=False
    )

    result = results[0]
    annotated_frame = result.plot()
    phone_count = len(result.boxes)

    if phone_count > 0:
        current_conf = float(result.boxes.conf.max())
        confidence_history.append(current_conf)
        smooth_conf = sum(confidence_history) / len(confidence_history)

        cv2.putText(
            annotated_frame,
            f"Confidence: {smooth_conf:.2f}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )
    else:
        confidence_history.clear()

    cv2.putText(
        annotated_frame,
        f"Detected Phones: {phone_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.imshow("Live Phone Detection - YOLO11n V2", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 8. Observations and Limitations

- Tested phone angles produced detections in the approximate 0.57–0.79 confidence range in observed webcam trials.
- Partial-phone detections were observed in some webcam tests.
- A face/glasses false positive was observed occasionally; false positives are not fully eliminated.
- One face-only negative image test returned zero phone detections; this is not sufficient to establish general false-positive performance.
- Confidence score is a model score, **not accuracy or probability of correctness**.
- A threshold of 0.50 filters predictions with confidence below 0.50. Higher thresholds may suppress genuine phones.
- Validation size is limited; the model is a college prototype, not production-grade proctoring evidence.

## 9. Conclusion

The V2 YOLO11n model is available for phone detection inference and webcam demonstration. It includes partial-phone annotations and negative images. It is suitable as a prototype component, with known limitations in false positives and generalization. Human review and broader testing are recommended before any consequential exam decision.